<a href="https://colab.research.google.com/github/jordynojeda/Fine-Tuned-LLM-with-Retrieval-Augmented-Generation-RAFT/blob/main/financial_advisor_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Fine-Tuning LLaMA to be a Financial Advisor (Supports LoRA and QLoRA)

This notebook demonstrates best practices for fine-tuning a LLaMA model using **LoRA** or **QLoRA** with [Unsloth](https://unsloth.ai/), based on the [financial-advisor-100 dataset](https://www.superteams.ai/blog/guide-to-fine-tune-your-llm-for-building-your-own-financial-advisor).

## Environment Set-up

In [ ]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install -q unsloth

## Load Base Model (LoRA vs QLoRA)


In [ ]:
from unsloth import FastLanguageModel
from google.colab import userdata
import torch

# Choose method: "LoRA" or "QLoRA"
fine_tuning_method = "QLoRA"

if fine_tuning_method == "LoRA":
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="meta-llama/Llama-3.2-3B",
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=False,
        token=userdata.get("HF_TOKEN")
    )
elif fine_tuning_method == "QLoRA":
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        max_seq_length=2048,
        dtype=torch.float16,
        load_in_4bit=True,
        token=userdata.get("HF_TOKEN")
    )


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.5: Fast Llama patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

# Chat Template Configuration

In [ ]:
# Set chat template for GGUF compatibility
chat_template = """
{{ bos_token }}
<|start_header_id|>system<|end_header_id|>
{{ system_message }}<|eot_id|>
{% for message in messages %}
<|start_header_id|>{{ message['role'] }}<|end_header_id|>
{{ message['content'] }}<|eot_id|>
{% endfor %}
<|start_header_id|>assistant<|end_header_id|>
"""

if tokenizer.chat_template != chat_template:
    tokenizer.chat_template = chat_template
    print("✅ Set custom chat_template for GGUF compatibility.")

✅ Set custom chat_template for GGUF compatibility.


## LoRA/QLoRA Configuration

In [ ]:
# LoRA fine-tuning configuration for FastLanguageModel using PEFT
# Reference: https://huggingface.co/docs/peft/v0.11.0/en/package_reference/lora#peft.LoraConfig

from peft import LoraConfig

# Target transformer modules to inject LoRA adapters into
# These are common projection layers in attention and MLP blocks of transformer models
target_modules = [
    "q_proj",     # Query projection (self-attention)
    "k_proj",     # Key projection
    "v_proj",     # Value projection
    "o_proj",     # Output projection from attention
    "gate_proj",  # Gating layer in FFN
    "up_proj",    # Feedforward network up-projection
    "down_proj"   # Feedforward network down-projection
]

# Flag to determine whether to include token embedding layer for training (e.g., when adding special tokens)
train_embeddings = False
if train_embeddings:
    target_modules.append("lm_head")  # lm_head is typically the output embedding layer

# LoRA configuration dictionary for clarity and reuse
lora_config = {
    "r": 16,  # Rank of the low-rank adapter matrices. Lower = smaller adapter, less compute.
              # According to the original LoRA paper, small r (e.g. 4–16) performs well.

    "target_modules": target_modules,  # List of layer names to apply LoRA to

    "lora_alpha": 16,  # Scaling factor for the LoRA weights.
                       # Larger alpha increases the impact of adapters (defaults often 16–64).

    "lora_dropout": 0.0,  # Dropout rate applied to the LoRA layers during training.
                          # Unsloth suggests 0.0 for best performance unless overfitting.

    "bias": "none",  # "none": no bias training; saves memory and computation.
                     # Alternatives: "all" or "lora_only" (rarely used in practice).

    "use_gradient_checkpointing": "unsloth",  # Activates gradient checkpointing. Saves VRAM by recomputing intermediate activations.
                                              # "unsloth" enables long-context training optimally.

    "random_state": 3407,  # Seed for reproducibility (affects adapter initialization).

    "use_rslora": False,  # Whether to scale LoRA weights by 1/sqrt(r) (recommended by HF in some cases).
                          # Often disabled when using tuned alpha directly.

    "loftq_config": None,  # Used only if integrating with LoftQ (quantization-aware LoRA).
                           # Leave as None for standard full-precision LoRA.
}

# Inject LoRA adapters into the base model using PEFT
model = FastLanguageModel.get_peft_model(
    model,
    **lora_config
)

# Confirm the configured target modules
print("LoRA modules applied to:", model.peft_config["default"].target_modules)


Unsloth 2025.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA modules applied to: {'up_proj', 'v_proj', 'o_proj', 'k_proj', 'gate_proj', 'down_proj', 'q_proj'}


## Load & Format Dataset

In [ ]:
from datasets import load_dataset

# Load the dataset (forcing redownload to avoid cache bugs)
dataset = load_dataset(
    "nihiluis/financial-advisor-100",
    split="train",
    download_mode="force_redownload"
)

print(dataset.column_names)

# Reformat to instruction-tuning format
def format_instruction(example):
    return {
        "instruction": example["question"],
        "input": "",  # optional input if needed
        "output": example["answer"]
    }

dataset = dataset.map(format_instruction)
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in ["instruction", "input", "output"]])

README.md:   0%|          | 0.00/539 [00:00<?, ?B/s]

(…)-00000-of-00001-f0708e72202ddcaa.parquet:   0%|          | 0.00/321k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100 [00:00<?, ? examples/s]

['id', 'question', 'answer', 'text']


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

## Tokenization

In [ ]:
# Tokenize each sample into prompt format
def tokenize(example):
    messages = [
        {"role": "system", "content": "You are a helpful and knowledgeable financial advisor specializing in providing clear, actionable advice to a wide range of financial questions."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return tokenizer(prompt, truncation=True, padding="max_length", max_length=2048)

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

## Training

In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./financial_llama",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="adamw_8bit" if fine_tuning_method == "QLoRA" else "adamw_torch",
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    seed=42,
    save_strategy="no",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

/tmp/ipython-input-7-2444576146.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 5 | Total steps = 65
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.768800
20,1.499900
30,1.437200
40,1.374500
50,1.244100
60,1.212500


TrainOutput(global_step=65, training_loss=1.403132754105788, metrics={'train_runtime': 2454.1538, 'train_samples_per_second': 0.204, 'train_steps_per_second': 0.026, 'total_flos': 4.6367955222528e+16, 'train_loss': 1.403132754105788, 'epoch': 5.0})

## Inference

In [ ]:
from transformers import TextStreamer

def ask_financial_question(question: str, max_new_tokens: int = 200):
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system", "content": "You are a helpful and knowledgeable financial advisor."},
        {"role": "user", "content": question}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer)
    print("📩 Answer:")
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=max_new_tokens)

ask_financial_question("How should I prioritize paying off debt vs investing?")

save_path = "./qlora-financial-advisor"
tokenizer.save_pretrained(save_path)

📩 Answer:
<|begin_of_text|>
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
<|eot_id|>
<|start_header_id|>system<|end_header_id|>
You are a helpful and knowledgeable financial advisor.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
How should I prioritize paying off debt vs investing?<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Prioritizing debt repayment vs. investing is a crucial decision that depends on your individual financial situation, goals, and risk tolerance. Here are some general guidelines:

1. **High-Interest Debt**: If you have high-interest debt (credit cards, personal loans), focus on paying these off as soon as possible. The interest rates are likely higher than what you could earn from investing, so it's generally a good idea to eliminate these debts first.

2. **Low-Interest Debt**: If you have low-interest debt (student loans, mortgages), you may want to consider investing while making regular payments. The interest rate on your debt is 

('./qlora-financial-advisor/tokenizer_config.json',
 './qlora-financial-advisor/special_tokens_map.json',
 './qlora-financial-advisor/chat_template.jinja',
 './qlora-financial-advisor/tokenizer.json')

## Save & Push

In [ ]:
if fine_tuning_method == "LoRA":

    model.save_pretrained(save_path)

    # Push to huggingface
    model.push_to_hub("jordynojeda/Llama-3.2-3B-financial-advisor-lora", token=userdata.get("HF_TOKEN"))
    tokenizer.push_to_hub("jordynojeda/Llama-3.2-3B-financial-advisor-lora", token=userdata.get("HF_TOKEN"))

elif fine_tuning_method == "QLoRA":

    model.save_pretrained(save_path)

    # Push to huggingface
    model.push_to_hub("jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora", token=userdata.get("HF_TOKEN"))
    tokenizer.push_to_hub("jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora", token=userdata.get("HF_TOKEN"))

In [ ]:
# Push to hugging face
model.push_to_hub_gguf(
    "jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF",
    tokenizer,
    quantization_method="q4_k_m",
    token=userdata.get("HF_TOKEN")
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 4.54 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 50%|█████     | 16/32 [00:01<00:01, 12.84it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [03:32<00:00,  6.64s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF/pytorch_model-00001-of-00004.bin...
Unsloth: Saving jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF/pytorch_model-00002-of-00004.bin...
Unsloth: Saving jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF/pytorch_model-00003-of-00004.bin...
Unsloth: Saving jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF/pytorch_model-00004-of-00004.bin...
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF into f16 GGUF format.
The output location will be /content/jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exp

Uploading...:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Saved GGUF to https://huggingface.co/jordynojeda/Meta-Llama-3.1-8B-Instruct-bnb-4bit-financial-advisor-qlora-GGUF


## ✅ Summary

- ✅ Supports both **LoRA** and **QLoRA** fine-tuning
- ✅ Based on Unsloth for efficient training
- ✅ Uses Hugging Face datasets and trainer for ease
- ✅ Outputs a Hugging Face-ready adapter model

Test and compare performance between `LoRA` and `QLoRA` by switching `fine_tuning_method` at the top.